In [ ]:
"""
Data Checking Code - Dataset Overview and Validation
Date: Nov 17 2025
Updated: June 29 2026

Input Files:
compilation_Venuti_2024_Serna_2021.csv

Purpose:
- Load and validate the stellar accretion dataset
- Count data points by spectral class (first letter of SpT)
- Count data points by mass bins (≤2.0 M☉ and >2.0 M☉)
- Count disk presence categories
- Count accretion status categories

This provides a quick overview of the dataset before running the main analysis.
"""

import pandas as pd
import os

def main():
    """Load and analyze the dataset."""
    
    # LOAD DATASET
    file_path = 'compilation_Venuti_2024_Serna_2021.csv'
    
    if not os.path.exists(file_path):
        print(f"Error: File '{file_path}' not found.")
        print(f"Current directory: {os.getcwd()}")
        return
    
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error loading file: {e}")
        return
    
    # Validate columns
    required_cols = ['Mstar', 'Disk', 'Acc']
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns: {missing_cols}")
    
    # Sort by mass then age
    sort_cols = [c for c in ['Mstar', 'logAge'] if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols).reset_index(drop=True)
    
    print("="*60)
    print("DATASET OVERVIEW AFTER INITIAL SORTING")
    print("="*60)
    print(f"Total datapoints: {len(df)}")
    
    # SPECTRAL CLASS COUNTS
    print("\n" + "-"*60)
    print("DATAPOINTS PER SPECTRAL CLASS (FIRST LETTER OF SpT)")
    print("-"*60)
    
    if 'SpT' in df.columns:
        df['SpT_class'] = df['SpT'].astype(str).str.strip().str[0]
        valid_classes = ['O', 'B', 'A', 'F', 'G', 'K', 'M']
        df['SpT_class'] = df['SpT_class'].where(
            df['SpT_class'].isin(valid_classes), 'Other'
        )
        df.loc[df['SpT'].isna(), 'SpT_class'] = 'NaN'
        spt_class_counts = df['SpT_class'].value_counts()
        
        print("OBAFGKM Classes:")
        for cls in valid_classes:
            print(f"  {cls}: {spt_class_counts.get(cls, 0)}")
        print(f"  NaN: {spt_class_counts.get('NaN', 0)}")
        print(f"  Other: {spt_class_counts.get('Other', 0)}")
    else:
        print("Warning: 'SpT' column not found")
    
    # MASS BIN COUNTS
    print("\n" + "-"*60)
    print("DATAPOINTS PER MASS BIN")
    print("-"*60)
    
    if 'Mstar' in df.columns:
        valid_mass = df['Mstar'].dropna()
        print(f"Valid Mstar entries: {len(valid_mass)} (NaN: {df['Mstar'].isna().sum()})")
        
        below_2 = valid_mass[valid_mass <= 2.0]
        above_2 = valid_mass[valid_mass > 2.0]
        
        print(f"Mstar ≤ 2.0 M☉ : {len(below_2)}")
        print(f"Mstar > 2.0 M☉ : {len(above_2)}")
        
        if len(valid_mass) > 0:
            print(f"\nMass Statistics:")
            print(f"  Min Mstar: {valid_mass.min():.3f} M☉")
            print(f"  Max Mstar: {valid_mass.max():.3f} M☉")
            print(f"  Mean Mstar: {valid_mass.mean():.3f} M☉")
            print(f"  Median Mstar: {valid_mass.median():.3f} M☉")
    else:
        print("Warning: 'Mstar' column not found")
    
    # DISK PRESENCE COUNTS
    print("\n" + "-"*60)
    print("DISK PRESENCE COUNTS")
    print("-"*60)
    
    if 'Disk' in df.columns:
        disk_counts = df['Disk'].value_counts(dropna=False)
        disk_map = {'y': 'Disk = Yes', 'n': 'Disk = No', 'y*': 'Disk = Maybe'}
        
        for key, label in disk_map.items():
            print(f"{label}: {disk_counts.get(key, 0)}")
        
        unexpected = [x for x in disk_counts.index if x not in disk_map and not pd.isna(x)]
        if unexpected:
            print(f"\nWarning: Unexpected disk values: {unexpected}")
            for val in unexpected:
                print(f"  {val}: {disk_counts.get(val, 0)}")
        
        if pd.isna(disk_counts.index).any():
            print(f"NaN: {disk_counts.get(pd.NA, 0)}")
    else:
        print("Warning: 'Disk' column not found")
    
    # ACCRETION STATUS COUNTS
    print("\n" + "-"*60)
    print("ACCRETION STATUS COUNTS")
    print("-"*60)
    
    if 'Acc' in df.columns:
        acc_counts = df['Acc'].value_counts(dropna=False)
        acc_map = {0: 'Accretion = No', 1: 'Accretion = Yes', 2: 'Accretion = Potential'}
        
        for key, label in acc_map.items():
            print(f"{label}: {acc_counts.get(key, 0)}")
        
        unexpected = [x for x in acc_counts.index if x not in acc_map and not pd.isna(x)]
        if unexpected:
            print(f"\nWarning: Unexpected accretion values: {unexpected}")
            for val in unexpected:
                print(f"  {val}: {acc_counts.get(val, 0)}")
        
        if pd.isna(acc_counts.index).any():
            print(f"NaN: {acc_counts.get(pd.NA, 0)}")
    else:
        print("Warning: 'Acc' column not found")
    
    # DATA COMPLETENESS SUMMARY
    print("\n" + "-"*60)
    print("DATA COMPLETENESS SUMMARY")
    print("-"*60)
    
    main_analysis_cols = ['logAge', 'Mstar', 'logMacc']
    missing_main = [col for col in main_analysis_cols if col not in df.columns]
    
    if not missing_main:
        complete_data = df.dropna(subset=main_analysis_cols)
        print(f"Complete rows for main analysis: {len(complete_data)}/{len(df)} ({len(complete_data)/len(df)*100:.1f}%)")
        
        print("\nMissing data per column:")
        for col in df.columns:
            missing = df[col].isna().sum()
            if missing > 0:
                print(f"  {col}: {missing} ({missing/len(df)*100:.1f}%)")
        
        # Additional completeness metrics
        print(f"\nColumns with complete data:")
        for col in df.columns:
            if df[col].isna().sum() == 0:
                print(f"  {col}")
    else:
        print(f"Warning: Missing required columns: {missing_main}")
    
    print("\nAnalysis complete.")

if __name__ == "__main__":
    main()

DATASET OVERVIEW AFTER INITIAL SORTING
Total datapoints: 1217

------------------------------------------------------------
DATAPOINTS PER SPECTRAL CLASS (FIRST LETTER OF SpT)
------------------------------------------------------------
OBAFGKM Classes:
  O: 6
  B: 45
  A: 77
  F: 112
  G: 137
  K: 487
  M: 101
  NaN: 251
  Other: 1

------------------------------------------------------------
DATAPOINTS PER MASS BIN
------------------------------------------------------------
Valid Mstar entries: 1128 (NaN: 89)
Mstar ≤ 2.0 M☉ : 944
Mstar > 2.0 M☉ : 184

Mass Statistics:
  Min Mstar: 0.149 M☉
  Max Mstar: 14.267 M☉
  Mean Mstar: 1.247 M☉
  Median Mstar: 0.901 M☉

------------------------------------------------------------
DISK PRESENCE COUNTS
------------------------------------------------------------
Disk = Yes: 147
Disk = No: 413
Disk = Maybe: 82

  n*: 155
  ?: 47
NaN: 0

------------------------------------------------------------
ACCRETION STATUS COUNTS
-------------------------